# Building GPT from First Principles - Little Shakespeare Training

**Complete implementation of GPT-style Transformer with detailed explanations**

This notebook demonstrates:
- Character-level tokenization
- Complete GPT architecture (all components)
- Training loop with proper optimization
- Text generation

**Structure**: Each section has explanations in markdown cells above the code cells.


## 1. Imports and Setup

**What we need:**
- `torch`: PyTorch for deep learning
- `torch.nn`: Neural network modules
- `torch.nn.functional`: Functional operations (softmax, etc.)
- `Dataset, DataLoader`: For data handling
- `math`: Mathematical functions (sqrt, exp, etc.)
- `requests`: To download dataset
- `Path`: File path handling


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math
import requests
from pathlib import Path


## 2. Character-Level Tokenizer

**Why Character-Level?**
- Task requirement: vocab_size ~65 (as per mentor's specification)
- Each character maps to unique integer ID
- Simple, no out-of-vocabulary (OOV) issues
- Model learns spelling, formatting, punctuation

**How it works:**
1. Find all unique characters in text
2. Create mappings: character → ID (encode) and ID → character (decode)
3. Vocabulary size = number of unique characters (~65 for English text)


In [ ]:
class CharTokenizer:
    """Character-level tokenizer for Shakespeare dataset (vocab_size ~65)"""
    
    def __init__(self, text):
        # Get all unique characters and sort them
        chars = sorted(list(set(text)))
        self.vocab_size = len(chars)
        
        # Create mappings: string to int (for encoding)
        self.stoi = {ch: i for i, ch in enumerate(chars)}  # string to int
        
        # Create mappings: int to string (for decoding)
        self.itos = {i: ch for i, ch in enumerate(chars)}  # int to string
        
    def encode(self, text):
        """Convert string to list of integers"""
        return [self.stoi.get(c, 0) for c in text]
    
    def decode(self, ids):
        """Convert list of integers to string"""
        return ''.join([self.itos[i] for i in ids])


## 3. Layer Normalization

**Mathematical Formulation:**
```
For input x ∈ ℝ^(batch × seq_len × d):
μ = (1/d) Σᵢ xᵢ           # Mean per feature dimension
σ² = (1/d) Σᵢ (xᵢ - μ)²  # Variance per feature dimension
x̂ = (x - μ) / √(σ² + ε)  # Normalized (ε prevents division by zero)
y = γ · x̂ + β            # Scale (γ) and shift (β) are learnable
```

**Why LayerNorm?**
- Stabilizes training by normalizing activations
- Reduces sensitivity to initialization
- Works better than BatchNorm for variable-length sequences
- Enables deeper networks (pre-norm architecture)

**Key Components:**
- `eps = 1e-5`: Small constant to prevent division by zero
- `scale (γ)`: Learnable parameter to scale normalized values
- `shift (β)`: Learnable parameter to shift normalized values


In [ ]:
class LayerNorm(nn.Module):
    """Layer Normalization: Normalize across feature dimension (per sample)"""
    
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5  # Small constant to prevent division by zero
        
        # Learnable parameters: γ (scale) and β (shift)
        self.scale = nn.Parameter(torch.ones(emb_dim))      # γ
        self.shift = nn.Parameter(torch.zeros(emb_dim))     # β
    
    def forward(self, x):
        """Normalize input: x → (x - μ) / σ → γ·x̂ + β"""
        # Compute mean and variance across last dimension (features)
        mean = x.mean(dim=-1, keepdim=True)      # μ
        var = x.var(dim=-1, keepdim=True, unbiased=False)  # σ²
        
        # Normalize: x̂ = (x - μ) / √(σ² + ε)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        
        # Scale and shift: y = γ·x̂ + β
        return self.scale * norm_x + self.shift


## 4. Causal Self-Attention

**Mathematical Formulation:**
```
1. Q = X · W_Q, K = X · W_K, V = X · W_V
2. S = QK^T / √d_k  (scaled dot-product attention scores)
3. M[i,j] = 0 if j≤i, -∞ if j>i  (causal mask)
4. A = softmax(S + M)  (attention weights, future positions = 0)
5. O = A · V  (weighted sum of values)
```

**Dimensions:**
- Input: x ∈ ℝ^(batch × seq_len × emb_dim)
- Output: context_vector ∈ ℝ^(batch × seq_len × out_dim)

**Key Insight:**
Each token attends only to previous tokens, enforcing autoregressive property:
- P(w_t | w_1, ..., w_{t-1}) instead of P(w_t | w_1, ..., w_n)
- This is essential for generation: during inference, future tokens don't exist!

**Why Scale by √d_k?**
- As embedding dimension grows, dot products become large
- Without scaling, softmax becomes very sharp (all probability on one token)
- Scaling prevents saturation and helps gradients flow better


In [ ]:
class CausalSelfAttention(nn.Module):
    """Causal self-attention with masking to prevent looking at future tokens"""
    
    def __init__(self, emb_dim, out_dim, context_length, drop_rate):
        super().__init__()
        # Linear projections: X → Q, K, V
        # W_Q, W_K, W_V ∈ ℝ^(emb_dim × out_dim)
        self.kw = nn.Linear(emb_dim, out_dim, bias=False)
        self.qw = nn.Linear(emb_dim, out_dim, bias=False)
        self.vw = nn.Linear(emb_dim, out_dim, bias=False)
        self.dropout = nn.Dropout(drop_rate)
        
        # Causal mask: M[i,j] = 1 if j > i (upper triangular)
        # After softmax with -∞: A[i,j] = 0 for j > i
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
    
    def forward(self, x):
        """Forward pass of causal self-attention"""
        b, t, emb_dim = x.shape
        
        # Step 1: Q = XW_Q, K = XW_K, V = XW_V
        key = self.kw(x)       # K ∈ ℝ^(b × t × out_dim)
        query = self.qw(x)     # Q ∈ ℝ^(b × t × out_dim)
        value = self.vw(x)     # V ∈ ℝ^(b × t × out_dim)
        
        # Step 2: S = QK^T (attention scores)
        # S[i,j] = similarity between token i (query) and token j (key)
        attn_score = query @ key.transpose(1, 2)  # ℝ^(b × t × t)
        
        # Step 3: Apply causal mask M: set future positions to -∞
        # After softmax: exp(-∞) = 0, so future positions have 0 attention weight
        attn_score.masked_fill_(self.mask.bool()[:t, :t], -torch.inf)
        
        # Step 4: Scale and softmax
        # Scale by √d_k prevents dot products from becoming too large
        # Softmax converts scores to probabilities: A[i,j] = P(j attends to i)
        attn_weight = torch.softmax(attn_score / math.sqrt(key.shape[-1]), dim=-1)
        attn_weight = self.dropout(attn_weight)
        
        # Step 5: Weighted sum O = AV
        # O[i] = Σ_j A[i,j] · V[j] (each output is weighted combination of values)
        context_vector = attn_weight @ value
        
        return context_vector


## 5. Multi-Head Attention

**Mathematical Formulation:**
```
Split d_model into h heads: d_k = d_model / h

For each head i:
  head_i = Attention(X · W_Q^i, X · W_K^i, X · W_V^i)

Concatenate: MultiHead = Concat(head₁, ..., head_h) · W_O
```

**Key Insight:**
Each head learns different attention patterns:
- **Head 1**: Previous token relationships (local dependencies)
- **Head 2**: Syntactic structures (grammar)
- **Head 3**: Semantic relationships (meanings)
- **Head 4**: Long-range dependencies (coreference)

**Benefits:**
- Richer representation than single head
- Same parameter count, multiple perspectives
- Parallel processing (no extra cost)
- Like having multiple experts, each with a specialty


In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Attention: Parallel attention heads for richer representation"""
    
    def __init__(self, emb_dim, out_dim, context_length, drop_rate, n_head):
        super().__init__()
        assert out_dim % n_head == 0, "out_dim must be divisible by n_head"
        
        # Split embedding dimension across heads
        head_dim = out_dim // n_head  # d_k = d_model / h
        
        # Create h independent attention heads
        # Each head processes same input but with different learned weights
        self.heads = nn.ModuleList([
            CausalSelfAttention(emb_dim, head_dim, context_length, drop_rate) 
            for _ in range(n_head)
        ])
        
    def forward(self, x):
        """Process input through all heads in parallel, then concatenate"""
        # Each head processes independently and learns different patterns
        head_outputs = [head(x) for head in self.heads]
        
        # Concatenate all head outputs along feature dimension
        # Result: ℝ^(batch × seq_len × out_dim)
        return torch.cat(head_outputs, dim=-1)
